In [ ]:
import os
import yaml
import json
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm

from anngeno import AnnGeno
import multiprocessing

num_cores = multiprocessing.cpu_count()
print(num_cores)

## Code

In [ ]:
def process_phenotypes_prs_long(
    gene_trait_df: pl.DataFrame,
    pheno_path: str,
    prs_path: str,
    cov_path: str,
    config: dict,
) -> pl.DataFrame:
    """
    Process phenotypes and PRS, compute residuals for each phenotype, and return a long-format Polars DataFrame.
    """

    # --- Step 1: Unique phenotypes ---
    unique_phenotypes = gene_trait_df['phenotype'].unique().to_list()
    if not unique_phenotypes:
        raise ValueError("No phenotypes found in gene_trait_df")

    # --- Step 2: Read data ---
    phenos = (
        pl.read_parquet(pheno_path)
        .rename({'IID': 'individual'})
        .select(['individual'] + unique_phenotypes)
        .drop_nulls()
    )

    prs_cols = [f"{pheno}_prs" for pheno in unique_phenotypes]
    prs = (
        pl.read_parquet(prs_path)
        .rename({'IID': 'individual'})
        .select(['individual'] + prs_cols)
        .drop_nulls()
    )

    cov_list = config.get("covariates", [])
    cov_df = (
        pl.read_parquet(cov_path)
        .rename({'sample': 'individual'})
        .select(['individual'] + cov_list)
        .with_columns(pl.col('individual').cast(pl.Int64))
    )

    # --- Step 3: Merge all into one DataFrame ---
    all_df = phenos.join(prs, on='individual', how='inner').join(
        cov_df, on='individual', how='inner'
    )
    all_pd = all_df.to_pandas()

    # --- Step 4: Compute residuals ---
    all_residuals_dfs = []

    for phenotype in unique_phenotypes:
        try:
            pheno_cols = [phenotype, f"{phenotype}_prs"] + cov_list
            temp_df = all_pd[['individual'] + pheno_cols].dropna()
            if len(temp_df) == 0:
                print(f"No data for phenotype: {phenotype}")
                continue

            y = temp_df[phenotype]
            X = temp_df.drop(columns=[phenotype, 'individual'])
            X = sm.add_constant(X)

            model = sm.OLS(y, X).fit()
            residuals = pd.Series(model.resid, index=temp_df.index, name=f"{phenotype}_residual")

            pheno_residuals = pd.concat([temp_df[['individual']], residuals], axis=1)
            all_residuals_dfs.append(pheno_residuals)

        except Exception as e:
            print(f"Error processing phenotype {phenotype}: {e}")
            continue

    if not all_residuals_dfs:
        raise ValueError("No residuals could be computed")

    # --- Step 5: Convert to long-format Polars DataFrame ---
    long_dfs = []
    for residual_df in all_residuals_dfs:
        p_wide = pl.DataFrame(residual_df).with_columns(
            pl.col('individual').cast(pl.String)
        )
        pheno_cols = [c for c in p_wide.columns if c.endswith('_residual')]
        pdf = (
            p_wide.unpivot(
                index=['individual'],
                on=pheno_cols,
                variable_name='phenotype',
                value_name='pheno_value',
            )
            .with_columns(
                pl.col('phenotype').str.replace('_residual', '').alias('phenotype')
            )
        )
        long_dfs.append(pdf)

    combined_pdf = pl.concat(long_dfs) if len(long_dfs) > 1 else long_dfs[0]

    return combined_pdf

def process_gene_genotypes(
    gene_id: str, 
    regions_dict: dict, 
    sample_list: list
) -> pl.DataFrame:
    """
    Extract genotypes for a specific gene
    """
    try:
        reg_dict = regions_dict[gene_id]
        geno = reg_dict['genotypes']
        
        # Find heterozygous genotypes (genotype == 1)
        rows, cols = np.where(geno == 1)
        het = pl.DataFrame({
            'id': np.array(reg_dict['variant_ids'])[rows],
            'individual': np.array(sample_list)[cols],
            'genotype': 1
        })
        
        # Find homozygous genotypes (genotype == 2)
        rows, cols = np.where(geno == 2)
        hom = pl.DataFrame({
            'id': np.array(reg_dict['variant_ids'])[rows],
            'individual': np.array(sample_list)[cols],
            'genotype': 2
        })
        
        geno_melt = pl.concat([het, hom]).with_columns(
            pl.lit(gene_id).alias('region')
        )
        
        return geno_melt
        
    except Exception as e:
        print(f"Error processing genotypes for gene {gene_id}: {e}")
        return None, None

def bootstrap_samples(
    gpa_df: pl.DataFrame, 
    n_bootstraps: int, 
    seed: int = None
) -> pl.DataFrame:
    """
    Bootstrap samples for correlation analysis
    """
    if seed is not None:
        np.random.seed(seed)

    # Get unique individuals
    unique_ids = gpa_df.select('individual').unique().to_series().to_list()
    n_ids = len(unique_ids)

    # Prepare all bootstrap samples
    sampled_ids = np.random.choice(unique_ids, size=(n_bootstraps, n_ids), replace=True)

    # Flatten and make DataFrame with bootstrap_id
    boot_id_col = np.repeat(np.arange(n_bootstraps), n_ids)
    sampled_flat = pl.LazyFrame({
        'bootstrap_id': boot_id_col,
        'individual': sampled_ids.ravel()
    })

    # Lazy join to replicate rows
    boot_df = sampled_flat.join(gpa_df.lazy(), on='individual', how='left')

    # Group by bootstrap_id + original grouping columns
    result = (
        boot_df
        .group_by(['bootstrap_id', 'id', 'phenotype', 'region', 'annotation'])
        .agg([
            pl.len().alias('n_individuals'),
            pl.col('pheno_value').mean().alias('mean_pheno_value'),
            pl.col('score').mean().alias('score'),
        ])
        .collect()
    )

    return result

In [ ]:
def corr_pipeline(
    gene_trait_df: pl.DataFrame,
    anngeno_path: str,
    pheno_path: str,
    prs_path: str,
    cov_path: str,
    exp_data_path: str,
    config: dict,
    maf: float = None,
    eur_samples_path: str = None,
    corr_method: str = "pearson",
    bootstrapping: bool = False,
    save_path: str = None,
):
    """
    Main processing pipeline for gene-trait correlation analysis.
    """

    # Process phenotypes (PRS + covariate correction)
    combined_pdf = process_phenotypes_prs_long(
        gene_trait_df,
        pheno_path,
        prs_path,
        cov_path,
        config,
    )

    # Initialize AnnGeno
    print("Loading AnnGeno...")
    ag = AnnGeno(anngeno_path, mode='r', low_mem=True)

    if maf:
        variants_to_keep = (
            ag.annotations.filter(pl.col('af_ukb') <= maf)
            .select('id')
            .collect()['id']
        )
        ag.subset_variants(set(variants_to_keep))

    eur_samples = None
    if eur_samples_path:
        eur_samples = pl.read_csv(eur_samples_path).with_columns(
            pl.col("eid").cast(pl.Utf8)
        )['eid'].to_list()
        ag.subset_samples(set(eur_samples))

    # Get regions
    unique_genes = gene_trait_df['gene_id'].unique().to_list()
    regions_dict = ag.get_many_regions(unique_genes)

    # Load Experimental assay scores
    print("Loading Experimental assay scores...")
    exp_df = pl.read_parquet(exp_data_path).filter(pl.col('region').is_in(unique_genes))

    # Collect annotation categories
    all_annotation_list = []
    rare_variant_annotations_dict = config.get("rare_variant_annotations")
    if rare_variant_annotations_dict:
        for category in rare_variant_annotations_dict.values():
            all_annotation_list.extend(category)

    all_results = []
    all_boot_results = []
    pheno_gis_df = None  # last computed pheno_gis_df, useful when plotting for a single gene-trait pair

    gene_to_traits = (
        gene_trait_df.group_by("gene_id")
        .agg(pl.col("phenotype").unique().alias("phenotypes"))
    )

    # --- Loop by gene, compute genotypes once ---
    for row in tqdm(gene_to_traits.iter_rows(named=True), total=len(gene_to_traits)):
        gene_id = row["gene_id"]
        phenotypes = row["phenotypes"]

        print(f"Processing gene {gene_id} with {len(phenotypes)} phenotypes")

        geno_melt = process_gene_genotypes(gene_id, regions_dict, ag.samples)
        if geno_melt is None:
            print(f"No genotype data for {gene_id}")
            continue

        anno_df = regions_dict[gene_id]["annotations"]
        exp_gene = exp_df.filter(pl.col("region") == gene_id).pivot(
            index=['mutant', 'region'],
            columns='assay_id',
            values='score'
        ).drop_nulls()

        # --- Loop over phenotypes for this gene ---
        for phenotype in phenotypes:
            print(f" -> {gene_id} - {phenotype}")

            pheno_data = combined_pdf.filter(pl.col("phenotype") == phenotype)
            if len(pheno_data) == 0:
                print(f"No phenotype data for {phenotype}")
                continue

            gp_df = geno_melt.join(pheno_data, on="individual")

            if eur_samples is not None:
                gp_df = gp_df.filter(pl.col("individual").is_in(eur_samples))

            gp_df = gp_df.filter(pl.col("genotype") == 1)
            if len(gp_df) == 0:
                print(f"No valid genotype-phenotype data for {gene_id} - {phenotype}")
                continue

            try:
                anno_melt = (
                    anno_df.join(exp_gene, on='mutant', how='inner')
                    .filter(pl.col('consequence_missense_variant') == 1)
                )
                available_annotations = list(set(all_annotation_list) & set(anno_melt.columns)) + exp_gene.columns[2:]
                if available_annotations:
                    anno_melt = anno_melt.unpivot(
                        index=['id', 'region', 'af_ukb'],
                        on=available_annotations,
                        variable_name='annotation',
                        value_name='score',
                    ).with_columns(
                        pl.col('score').cast(pl.Float32).alias('score')
                    )
                else:
                    print(f"No valid annotations for {gene_id}")

                ## Add all variant scores
                annos_all_vars = list(set(all_annotation_list) & set(anno_df.columns))
                all_anno_melt = anno_df.filter(pl.col('consequence_missense_variant') == 1).unpivot(
                    index=['id', 'region', 'af_ukb'],
                    on=annos_all_vars,
                    variable_name='annotation',
                    value_name='score',
                ).with_columns(
                    (pl.col('annotation') + "_allvars").alias('annotation')
                )

                anno_join = pl.concat([anno_melt, all_anno_melt])

                gpa_df = gp_df.join(anno_join, on='id', how='inner')
                if len(gpa_df) == 0:
                    print(f"No data after joining annotations for {gene_id} - {phenotype}")
                    continue
                
                # --- correlation step ---
                pheno_gis_df = (
                    gpa_df.group_by(['id', 'phenotype', 'region', 'annotation'])
                    .agg([
                        pl.len().alias('n_individuals'),
                        pl.col('pheno_value').mean().alias('mean_pheno_value'),
                        pl.col('score').mean().alias('score'),
                    ])
                    .drop_nulls(subset=['mean_pheno_value', 'score'])
                )
                if len(pheno_gis_df) == 0:
                    continue

                corr_df = (
                    pheno_gis_df.group_by(['annotation', 'phenotype', 'region'])
                    .agg(pl.corr('score', 'mean_pheno_value', method=corr_method).alias('corr'))
                    .with_columns(pl.col('corr').abs().alias('abs_corr'))
                    .filter(pl.col('abs_corr').is_not_null())
                    .sort('abs_corr', descending=True)
                    .with_columns([
                        pl.lit(gene_id).alias('gene_id'),
                        pl.lit(phenotype).alias('phenotype_name'),
                    ])
                )

                all_results.append(corr_df)

                if bootstrapping:
                    print(f"Running bootstrap for {gene_id} - {phenotype}")
                    boot_df = bootstrap_samples(gpa_df, n_bootstraps=1000, seed=42)
                    boot_corr_df = (
                        boot_df.group_by(['phenotype', 'region', 'annotation', 'bootstrap_id'])
                        .agg(pl.corr('score', 'mean_pheno_value', method=corr_method).alias('corr'))
                        .with_columns(pl.col('corr').abs().alias('abs_corr'))
                        .sort('abs_corr', descending=True)
                        .drop_nans()
                        .with_columns([
                            pl.lit(gene_id).alias('gene_id'),
                            pl.lit(phenotype).alias('phenotype_name'),
                        ])
                    )
                    all_boot_results.append(boot_corr_df)

            except Exception as e:
                print(f"Error processing annotations for {gene_id} - {phenotype}: {e}")
                continue

    # Combine results
    final_corr_df = pl.concat(all_results)
    print(f"Final correlation results shape: {final_corr_df.shape}")
    if save_path:
        final_corr_df.write_parquet(f'{save_path}/multi_gene_trait_correlations.parquet')

    final_boot_df = None
    if bootstrapping:
        final_boot_df = pl.concat(all_boot_results)
        print(f"Final bootstrap results shape: {final_boot_df.shape}")
        if save_path:
            final_boot_df.write_parquet(f'{save_path}/multi_gene_trait_bootstrap_correlations.parquet')

    return pheno_gis_df, final_corr_df, final_boot_df


## Run code

### Define params

In [ ]:
# Configuration and paths
corr_method = 'spearman'
bootstrapping = False
trait_type = 'quantitative'

config_path = f'/home/dnanexus/ukbgym/config.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

pheno_path = '/home/dnanexus/data_dir/phenotypes/phenotypes190_missing20_unique2_int.parquet'
prs_path = '/home/dnanexus/data_dir/phenotypes/PRS190_EUR_missing20_unique2_int.parquet'
cov_path = '/home/dnanexus/data_dir/phenotypes/250709_quant_phenotypes_covariates_genetic_pcs_prs_corrected.parquet'

anngeno_path = '/home/dnanexus/data_dir/dms_coding.ag'
eur_samples_path = '/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv'
save_path = None

In [ ]:
mac = 10
n = !wc -l $eur_samples_path
n_samples = int(n[0].split(' ')[0])
print(n_samples)

maf = mac/(2*n_samples)
maf = 0.001 # Manually set MAF
maf

### Find more gene-trait assocs

In [ ]:
# pgym = pl.read_parquet('/home/dnanexus/data_dir/exp_data/250717_proteingym_SNP_DMS_scores_human_coding_genes.parquet').rename(
#     {'file_name':'assay_description',
#      'dms_score': 'score'}
# ).with_columns(
#     pl.lit('DMS').alias('assay_name'),
#     pl.lit('proteinGym').alias('source')
# ).select(['source', 'assay_description', 'assay_name', 'gene_name', 'region', 'mutant', 'score'])
# print(pgym.head(2))

# marsh = pl.read_parquet('/home/dnanexus/data_dir/exp_data/250822_Livesey_Marsh_36DMS.parquet').rename(
#     {'fitness_assay': 'assay_description',
#     'dms_type': 'assay_name',
#     'dms_score': 'score'}
# ).with_columns(
#     pl.lit('Livesey2025').alias('source')
# ).select(['source', 'assay_description', 'assay_name', 'gene_name', 'region', 'mutant', 'score'])
# print(marsh.head(2))

# beltran = pl.read_parquet('/home/dnanexus/data_dir/exp_data/250819_Beltran_ssm_scores.parquet').rename(
#     {'fitness': 'score'}
# ).with_columns(
#     pl.lit('aPCA').alias('assay_name'),
#     pl.lit('Beltran2025_abundance_protein_fragment_complementation_assay_aPCA').alias('assay_description'),
#     pl.lit('Beltran2025').alias('source')
# ).select(['source', 'assay_description', 'assay_name', 'gene_name', 'region', 'mutant', 'score'])
# print(beltran.head(2))

# exp_df = pl.concat([pgym, marsh, beltran]).with_columns(
#     (pl.col('assay_name') + '_' + pl.col('source') + '_' + pl.col('assay_description').str.replace(" ", "_")).alias('assay_id')
# )
# exp_df.write_parquet('/home/dnanexus/data_dir/exp_data/250825_combinedAssays_proteinGym_livesey_beltran.parquet')

exp_df = pl.read_parquet('/home/dnanexus/data_dir/exp_data/250825_combinedAssays_proteinGym_livesey_beltran.parquet')
exp_df

In [ ]:
exp_df = pl.concat([pgym, marsh, beltran]).with_columns(
    (pl.col('assay_name') + '_' + pl.col('source') + '_' + pl.col('assay_description').str.replace(" ", "_")).alias('assay_id')
)
# exp_df.write_parquet('/home/dnanexus/data_dir/exp_data/250825_combinedAssays_proteinGym_livesey_beltran.parquet')
exp_df

In [ ]:
gcts = exp_df[['assay_id', 'gene_name']].unique()['gene_name'].value_counts(sort=True).filter(pl.col('count') > 1)
gcts

In [ ]:
gb = pl.read_parquet('/home/dnanexus/data_dir/genebass_continuous_associations_ukbbgym.pq').filter(pl.col('gene_symbol').is_in(gcts))
gb[['gene_symbol', 'description']].unique()
# gb[['gene_symbol']].unique()

In [ ]:
exp_df.drop_nulls()[['assay_id', 'gene_name']].unique()['gene_name'].value_counts(sort=True).filter(pl.col('gene_name').is_in(gb['gene_symbol'].unique().to_list()))

In [ ]:
a = exp_df.drop_nulls().filter(pl.col('gene_name')=='ASPA').pivot(
    index=['mutant', 'region', 'gene_name'],
    on='assay_id',
    values='score'
)#.drop_nulls()
a

In [ ]:
a.select([
    pl.col(col).is_null().sum().alias(col)
    for col in a.columns
])

In [ ]:
# exp_df.filter(pl.col('gene_name') == 'ASPA').filter(pl.col('assay_id').str.contains('00000657')).write_parquet('/home/dnanexus/data_dir/exp_data/250825_ASPA_Livesey_2DMS.parquet')

### Get DMS assays for gene

In [ ]:
exp_df = pl.read_parquet('/home/dnanexus/data_dir/exp_data/250825_combinedAssays_proteinGym_livesey_beltran.parquet')
exp_df

In [ ]:
gcts = exp_df[['assay_id', 'gene_name']].unique()['gene_name'].value_counts(sort=True)
gcts

In [ ]:
rvat = pl.read_parquet('/home/dnanexus/data_dir/rvat_EUR_500k_regenie.parquet').with_columns(
    pl.when(pl.col('trait_type') == 'quantitative')
    .then(pl.col('trait') + '_int')
    .otherwise(pl.col('trait'))
    .alias('phenotype')
)#.filter(pl.col('trait')!='seated_height')

gc_rvat = gcts.join(rvat, on='gene_name')
gc_rvat.filter(pl.col('bonf_significance')==True).sort(by='count', descending=True).filter(pl.col('count') > 1).filter(pl.col('trait_type') == 'quantitative') #[['gene_name', 'count']].unique().sort('count', descending=True)

In [ ]:
gene_trait_df = rvat.filter(pl.col('trait_type')=='quantitative').filter(pl.col('gene_name').is_in(['ASPA'])).filter(pl.col('phenotype')=='forced_vital_capacity_fvc_int')
gene_trait_df

## Exectute pipeline

In [ ]:
gene_trait_df = pl.DataFrame({
    'gene_id': ["ENSG00000108381"],#["ENSG00000141867"],
    'phenotype': ["forced_vital_capacity_fvc"] #['jurgens_bipolar_disorder']
})

In [ ]:
# exp_data_path = '/home/dnanexus/data_dir/exp_data/250825_combinedAssays_proteinGym_livesey_beltran.parquet'
exp_data_path = '/home/dnanexus/data_dir/exp_data/250825_ASPA_Livesey_2DMS.parquet'

plot_df, corr_df, boot_df = corr_pipeline(
    gene_trait_df=gene_trait_df,
    anngeno_path=anngeno_path,
    pheno_path=pheno_path,
    prs_path=prs_path,
    cov_path=cov_path,
    exp_data_path=exp_data_path,
    config=config,
    maf=maf,
    eur_samples_path=eur_samples_path,
    corr_method=corr_method,
    bootstrapping=True,
    save_path=save_path
)

boot_df

In [ ]:
plot_df['annotation'].value_counts(sort=True)

In [ ]:
from plotnine import *

all_vars = False
if all_vars:
    med_df = corr_df.drop_nans().group_by("annotation").agg([
        pl.col("abs_corr").median().alias("median_abs_corr")
    ])
else:
    med_df = corr_df.filter(~pl.col('annotation').str.contains('_allvars')).drop_nans().group_by("annotation").agg([
        pl.col("abs_corr").median().alias("median_abs_corr")
    ])

corr_pd = med_df.join(corr_df, on='annotation').to_pandas()

corr_pd['annotation'] = pd.Categorical(
    corr_pd['annotation'],
    categories=corr_pd.sort_values('median_abs_corr', ascending=False)['annotation'].unique(),
    ordered=True
)

# Plot
(
    ggplot(corr_pd, aes(x='annotation', y='abs_corr', fill='annotation')) +
    geom_bar(stat='identity', alpha=0.8) +
    theme_minimal() +
    coord_flip() +
    labs(
        x='Annotation',
        y='Absolute Spearman correlation'
    ) +
    theme(
        figure_size=(8, 4),
        legend_position='none'
    )
)

In [ ]:
from plotnine import *

summary_df = (
    boot_df
    .group_by("annotation")
    .agg([
        pl.col("abs_corr").mean().alias("mean_abs_corr"),
        pl.col("abs_corr").std().alias("se_abs_corr"),
        pl.col("abs_corr").quantile(0.025).alias("quant_low"),
        pl.col("abs_corr").quantile(0.975).alias("quant_high"),
    ])
    .with_columns([
        (pl.col("mean_abs_corr") - 1.96*pl.col("se_abs_corr")).alias("ci_low"),
        (pl.col("mean_abs_corr") + 1.96*pl.col("se_abs_corr")).alias("ci_high")
    ])
    .to_pandas()
)

# Ordering by mean correlation
summary_df['annotation'] = pd.Categorical(
    summary_df['annotation'],
    categories=summary_df.sort_values('mean_abs_corr', ascending=False)['annotation'],
    ordered=True
)

summary_df['color_dms'] = summary_df['annotation'].str.startswith('dms_').map({True: 'DMS', False: 'Other'})

# Plot
(
    ggplot(summary_df, aes(x='annotation', y='mean_abs_corr', fill='color_dms')) +
    geom_col(alpha=0.8) +
    # geom_errorbar(aes(ymin='ci_low', ymax='ci_high'), width=0.2) +
    geom_errorbar(aes(ymin='quant_low', ymax='quant_high'), width=0.2) +
    coord_flip() +
    theme_538() +
    labs(
        x='Annotation',
        y='Mean absolute correlation (95% CI)'
    ) +
    theme(
        figure_size=(8, 3),
        legend_position='none'
    )
)

## Single gene-trait plot

In [ ]:
corr_df.drop_nans().filter(pl.col('annotation')=='am_pathogenicity').sort('abs_corr', descending=True)#[0]['phenotype'].item()

In [ ]:
gene_trait_df = pl.DataFrame({
    'gene_id': ["ENSG00000163554"],#["ENSG00000141867"],
    'phenotype': ["mean_reticulocyte_volume_int"] #['jurgens_bipolar_disorder']
})

plot_df_gene, corr_results_gene, _ = corr_pipeline(
    gene_trait_df=gene_trait_df,
    anngeno_path=anngeno_path,
    pheno_path=pheno_path,
    prs_path=prs_path,
    cov_path=cov_path,
    exp_data_path=exp_data_path,
    config=config,
    maf=maf,
    eur_samples_path=eur_samples_path,
    corr_method=corr_method,
    bootstrapping=bootstrapping,
    save_path=save_path
)

plot_df_gene

In [ ]:
import sys
from IPython.display import display

def plot_correlation(plot_df, phenotype, gene_id, annotation, method='spearman'):
    # Filter and add ranks
    df_filtered = plot_df.filter(
        (pl.col('phenotype') == phenotype) &
        (pl.col('region') == gene_id) &
        (pl.col('annotation') == annotation)
    ).with_columns([
        pl.col('score').rank().alias('score_rank'),
        pl.col('mean_pheno_value').rank().alias('pheno_rank')
    ])

    # Determine columns to correlate and plot
    if method.lower() == 'spearman':
        x_col, y_col = 'score_rank', 'pheno_rank'
    elif method.lower() == 'pearson':
        x_col, y_col = 'mean_score', 'mean_pheno_value'
    else:  # Pearson
        sys.exit(f"Unrecognized method. Use 'spearman' or 'pearson'.")

    # Compute correlation using Polars
    corr = df_filtered.select([pl.corr(x_col, y_col, method='pearson')]).to_numpy()[0, 0]

    corr_text = f"{method.title()} r = {corr:.2f}"

    # Build plot
    rc_plot = (
        ggplot(df_filtered.to_pandas(), aes(x=x_col, y=y_col)) +
        geom_point(alpha=0.25) +
        geom_smooth(method='lm', se=True, color='darkred') +
        theme_minimal() +
        labs(
            x=f"{annotation} {'rank' if method.lower()=='spearman' else ''}",
            y=f"{phenotype} residual {'rank' if method.lower()=='spearman' else ''}"
        ) +
        annotate(
            'text',
            x=df_filtered[x_col].min(),
            y=df_filtered[y_col].max(),
            label=corr_text,
            ha='left',
            va='top',
            size=12
        ) +
        theme(figure_size=(5, 4))
    )

    return rc_plot

In [ ]:
corr_method = 'spearman'
am_plot = plot_correlation(plot_df_gene, gene_trait_df['phenotype'].item(), gene_trait_df['gene_id'].item(), 'am_pathogenicity', method=corr_method)

display(am_plot)

In [ ]:
plot_df_gene['annotation'].value_counts(sort=True)